## Silver Layer — Clean, Deduplicate, Enrich
**Reads from:** `main.bronze.*`
**Writes to:** `main.silver.*` (Delta) + Lakebase OLTP tables

Transformations applied:
- Deduplicate by natural key (keep latest per ticker/article)
- Normalize exchange codes (XNAS → NASDAQ)
- Derive new columns (daily_return, price_range, article_age_days)
- Validate data (filter nulls, invalid prices)
- Sync cleaned data to Lakebase PostgreSQL tables


In [ ]:
# 0. Imports and config
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from databricks.sdk import WorkspaceClient
from datetime import datetime

spark = SparkSession.builder.getOrCreate()

PROCESSED_AT = datetime.now().isoformat()
print(f"Silver transformation started: {PROCESSED_AT}")


In [ ]:
# 1. Setup Silver schema
spark.sql("CREATE SCHEMA IF NOT EXISTS main.silver")
print("Schema main.silver ready")


In [ ]:
# 2. Silver Companies
# - Deduplicate: keep latest record per ticker
# - Normalize: exchange codes to readable names
# - Enrich: add market_cap_billions
print("\n--- Processing companies ---")

bronze_companies = spark.table("main.bronze.raw_companies")

window_ticker = Window.partitionBy("ticker").orderBy(F.col("ingested_at").desc())

silver_companies = (
    bronze_companies
    .withColumn("row_num", F.row_number().over(window_ticker))
    .filter(F.col("row_num") == 1)
    .drop("row_num", "raw_json", "batch_id")
    .withColumn("exchange_name",
        F.when(F.col("exchange") == "XNAS", "NASDAQ")
         .when(F.col("exchange") == "XNYS", "NYSE")
         .when(F.col("exchange") == "XASE", "AMEX")
         .when(F.col("exchange") == "ARCX", "NYSE Arca")
         .otherwise(F.col("exchange"))
    )
    .withColumn("market_cap_billions",
        F.round(F.col("market_cap") / 1e9, 2)
    )
    .withColumn("processed_at", F.lit(PROCESSED_AT))
)

(silver_companies
 .write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("main.silver.companies"))

count = spark.table("main.silver.companies").count()
print(f"Written {count} rows → main.silver.companies ✓")

print("\nSample:")
spark.table("main.silver.companies") \
     .select("ticker", "name", "exchange_name", "market_cap_billions", "sic_description") \
     .show(5, truncate=False)


In [ ]:
# 3. Silver Price Snapshots
# - Deduplicate: keep latest per (ticker, snapshot_date)
# - Derive: daily_return_pct, price_range, is_up_day
# - Validate: filter rows where close <= 0
print("\n--- Processing price snapshots ---")

bronze_prices = spark.table("main.bronze.raw_price_snapshots")

window_price = Window.partitionBy("ticker", "snapshot_date").orderBy(F.col("ingested_at").desc())

silver_prices = (
    bronze_prices
    .withColumn("row_num", F.row_number().over(window_price))
    .filter(F.col("row_num") == 1)
    .drop("row_num", "raw_json", "batch_id", "timestamp_ms")
    .filter(F.col("close") > 0)
    .withColumn("daily_return_pct",
        F.round(((F.col("close") - F.col("open")) / F.col("open")) * 100, 4)
    )
    .withColumn("price_range",
        F.round(F.col("high") - F.col("low"), 4)
    )
    .withColumn("is_up_day",
        F.col("close") >= F.col("open")
    )
    .withColumn("open",  F.round("open",  4))
    .withColumn("high",  F.round("high",  4))
    .withColumn("low",   F.round("low",   4))
    .withColumn("close", F.round("close", 4))
    .withColumn("vwap",  F.round("vwap",  4))
    .withColumn("processed_at", F.lit(PROCESSED_AT))
)

(silver_prices
 .write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("main.silver.price_snapshots"))

count = spark.table("main.silver.price_snapshots").count()
print(f"Written {count} rows → main.silver.price_snapshots ✓")

print("\nSample:")
spark.table("main.silver.price_snapshots") \
     .select("ticker", "snapshot_date", "open", "close", "daily_return_pct", "price_range", "is_up_day") \
     .show(5)


In [ ]:
# 4. Silver News Articles
# - Deduplicate: keep latest per article_id
# - Parse: published_utc → timestamp
# - Derive: article_age_days, description_length, has_sentiment
# - Validate: filter null titles
print("\n--- Processing news articles ---")

bronze_news = spark.table("main.bronze.raw_news_articles")

window_news = Window.partitionBy("article_id").orderBy(F.col("ingested_at").desc())

silver_news = (
    bronze_news
    .filter(F.col("article_id").isNotNull())
    .filter(F.col("title").isNotNull())
    .withColumn("row_num", F.row_number().over(window_news))
    .filter(F.col("row_num") == 1)
    .drop("row_num", "raw_json", "batch_id")
    .withColumn("published_ts",
        F.to_timestamp(F.col("published_utc"))
    )
    .withColumn("article_age_days",
        F.datediff(F.current_date(), F.to_date(F.col("published_utc")))
    )
    .withColumn("description_length",
        F.length(F.col("description"))
    )
    .withColumn("has_sentiment",
        F.col("sentiment").isNotNull()
    )
    .withColumn("sentiment",
        F.lower(F.col("sentiment"))
    )
    .withColumn("processed_at", F.lit(PROCESSED_AT))
)

(silver_news
 .write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("main.silver.news_articles"))

count = spark.table("main.silver.news_articles").count()
print(f"Written {count} rows → main.silver.news_articles ✓")

print("\nSample:")
spark.table("main.silver.news_articles") \
     .select("ticker", "title", "publisher_name", "sentiment", "article_age_days", "has_sentiment") \
     .show(5, truncate=True)


In [ ]:
# 5. Sync to Lakebase OLTP tables
# Uses fresh OAuth token at runtime — no expiry issues
print("\n--- Syncing to Lakebase OLTP tables ---")

try:
    import psycopg2
    import psycopg2.extras

    # Load static connection details from secrets
    lb_host   = dbutils.secrets.get(scope="capstone", key="lakebase_host")
    lb_user   = dbutils.secrets.get(scope="capstone", key="lakebase_user")
    lb_dbname = dbutils.secrets.get(scope="capstone", key="lakebase_dbname")

    # Generate fresh OAuth token at runtime (never expires mid-run)
    w   = WorkspaceClient()
    tok = w.config.token

    conn = psycopg2.connect(
        host    = lb_host,
        port    = 5432,
        user    = lb_user,
        password= tok,
        dbname  = lb_dbname,
        sslmode = "require"
    )
    cur = conn.cursor()
    print("Lakebase connection established ✓")

    # -- Upsert companies --
    companies_data = [
        (row.ticker, row.name, row.sic_description, None,
         row.exchange_name, row.market_cap, row.description, PROCESSED_AT)
        for row in spark.table("main.silver.companies").collect()
    ]
    psycopg2.extras.execute_values(cur, """
        INSERT INTO stock_assistant.companies
            (ticker, name, sector, industry, exchange, market_cap, description, updated_at)
        VALUES %s
        ON CONFLICT (ticker) DO UPDATE SET
            name        = EXCLUDED.name,
            sector      = EXCLUDED.sector,
            exchange    = EXCLUDED.exchange,
            market_cap  = EXCLUDED.market_cap,
            description = EXCLUDED.description,
            updated_at  = EXCLUDED.updated_at
    """, companies_data)
    print(f"  Upserted {len(companies_data)} rows → lakebase companies ✓")

    # -- Upsert price_snapshots --
    prices_data = [
        (row.ticker, row.open, row.high, row.low, row.close,
         int(row.volume) if row.volume else None, row.snapshot_date)
        for row in spark.table("main.silver.price_snapshots").collect()
    ]
    psycopg2.extras.execute_values(cur, """
        INSERT INTO stock_assistant.price_snapshots
            (ticker, open, high, low, close, volume, snapshot_ts)
        VALUES %s
        ON CONFLICT (ticker, snapshot_ts) DO NOTHING
    """, prices_data)
    print(f"  Upserted {len(prices_data)} rows → lakebase price_snapshots ✓")

    # -- Upsert news_articles --
    news_data = [
        (row.ticker, row.title, row.publisher_name,
         row.article_url, row.description, row.published_ts)
        for row in spark.table("main.silver.news_articles").collect()
        if row.published_ts is not None
    ]
    psycopg2.extras.execute_values(cur, """
        INSERT INTO stock_assistant.news_articles
            (ticker, headline, source, url, body, published_at)
        VALUES %s
        ON CONFLICT DO NOTHING
    """, news_data)
    print(f"  Upserted {len(news_data)} rows → lakebase news_articles ✓")

    conn.commit()
    cur.close()
    conn.close()
    print("\nLakebase sync complete ✓")

except Exception as e:
    print(f"\nLakebase sync error: {e}")
    print("Check that secrets are stored and Lakebase project is active")


In [ ]:
# 6. Silver summary
print("\n=== Silver Transformation Summary ===")
print(f"Processed at: {PROCESSED_AT}\n")

for table in ["companies", "price_snapshots", "news_articles"]:
    try:
        count = spark.table(f"main.silver.{table}").count()
        print(f"  main.silver.{table:<20} rows: {count:>6}")
    except Exception as e:
        print(f"  main.silver.{table:<20} ERROR: {e}")

print("\nSilver transformation complete ✓")
